In [1]:
#!/usr/bin/env python3
import subprocess
import glob
import os
import sys

In [ ]:
# create env
os.makedirs(FASTQC_RAW, exist_ok=True)
os.makedirs(TRIMMED, exist_ok=True)
os.makedirs(FASTQC_TRIMMED, exist_ok=True)

In [ ]:
# raw data
raw_files = glob.glob("*.fastq.gz")
if not raw_files:
    sys.exit("ERROR: No raw fastq.gz files found")

subprocess.run(["fastqc", *raw_files, "-o", FASTQC_RAW, "--quiet"], check=True)

In [ ]:
# cut adapt
for i in samples:
  
    r1_in, r2_in = f"{i}_S{i}_L001_R1_001.fastq.gz", f"{i}_S{i}_L001_R2_001.fastq.gz"
    
    r_out = [f"{TRIMMED}/{i}_S{i}_L001_R1_paired.fastq.gz",
             f"{TRIMMED}/{i}_S{i}_L001_R1_unpaired.fastq.gz",
             f"{TRIMMED}/{i}_S{i}_L001_R2_paired.fastq.gz",
             f"{TRIMMED}/{i}_S{i}_L001_R2_unpaired.fastq.gz"]

    cmd = [
        "trimmomatic", "PE", "-threads", "2",
        r1_in, r2_in,
        *r_out,
        f"ILLUMINACLIP:{ADAPTERS}/TruSeq3-PE.fa:2:30:10:8:true",
        "SLIDINGWINDOW:4:20",
        "MINLEN:30"
    ]

    subprocess.run(cmd, check=True)

In [ ]:
# fastqc
trimmed_files = glob.glob(f"{TRIMMED}/*_paired.fastq.gz")
subprocess.run(["fastqc", *trimmed_files, "-o", FASTQC_TRIMMED, "-t", "4", "--quiet"], check=True)

In [ ]:
# index
subprocess.run(["bwa", "index", GENOME], check=True)

In [ ]:
for i in samples:
    r1 = f"{TRIMMED}/{i}_S{i}_L001_R1_paired.fastq.gz"
    r2 = f"{TRIMMED}/{i}_S{i}_L001_R2_paired.fastq.gz"
    bam = f"{i}.sorted.bam"

    # bwa mem
    bwa = subprocess.Popen(
        ["bwa", "mem", "-t", "8", "-M", GENOME, r1, r2],
        stdout=subprocess.PIPE
    )

    # samtools 
    view = subprocess.Popen(
        ["samtools", "view", "-b", "-"],
        stdin=bwa.stdout,
        stdout=subprocess.PIPE
    )

    # sort 
    sort = subprocess.Popen(
        ["samtools", "sort", "-@","4", "-o", bam],
        stdin=view.stdout
    )

    # stdout from PIPE
    bwa.stdout.close()
    view.stdout.close()

In [ ]:
# index
subprocess.run(["samtools", "index", f"{i}.sorted.bam"], check=True)